# Différentes approches numériques de la suite de Fibonnaci - Grand oral
Fait à l'aide de: [CP Algorithms - Fibonnaci numbers](https://cp-algorithms.com/algebra/fibonacci-numbers.html#closed-form-expression)

Différentes manières de calculer la plus grandes valeur de Fibonnacci, le plus rapidement possible sur ordinateur
### 1ère approche - version naïve
Voici la définition de la suite de Fibonnaci:
$F_0 = 0, F_1 = 1$
$F_n = F_{n-1} + F_{n-2}$
Si on reprend cette définition et qu'on essaie de l'implémenter en Python:

```python
def fib_naive(x):
  if x == 0 or x==1: return x
  return fib_naive(x-1)+fib_naive(x-2)
```

Mais si on essaie de calculer une valeur relativement grande comme `fib(100)` par exemple, cela peut prendre énormément de temps !

<faire schéma arbre>, on pourrait réutiliser ces valeurs pour ne pas refaire les calculs bêtement (LookUp Table): 
```python
lut = {
	0: 0,
	1: 1
}
def fib_lut(x):
  if lut.get(x) != None: return lut[x]
  r = fib_lut(x-1)+fib_lut(x-2)
  lut[x] = r
  return r
```
On peut aussi le calculer comme on ferait à la main:
```python
def fib_iter(x):
  a = 0
  b = 1
  for i in range(x): 
    tmp = a + b
    a = b
    b = tmp
  return a;
```
On commence à arriver à de très grands nombres, si vous ne le faites pas déjà (python le fait très bien, avec pleins d'optimisations), l'algorithme de multiplication de Karatsuba peut nous aider de passer de la multiplication naïve $O(n²)$ à $O(n^{log(2)})$ !

### Multiplication de matrices
Essayons une autre approche:
$$\begin{pmatrix}
1 & 1 \\
1 & 0
\end{pmatrix}\begin{pmatrix}
F_{n} \\
F_{n-1}
\end{pmatrix}=\begin{pmatrix}
F_{n}+F_{n-1} \\
F_{n}
\end{pmatrix}=\begin{pmatrix}
F_{n+1} \\
F_{n}
\end{pmatrix}$$
On peut donc multiplier des matrices pour calculer des éléments de la suite de Fibonnaci !
On peut ensuite démontrer avec une preuve par récurrence \<oral>:
$$\begin{pmatrix}
1 & 1 \\
1 & 0
\end{pmatrix}^n
\begin{pmatrix}
F_1 \\
F_0
\end{pmatrix}
=
\begin{pmatrix}
F_{n+1}  \\
F_{n}
\end{pmatrix}$$
Et puisque: $$\begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix}
= \begin{pmatrix} F_2 & F_1 \\ F_1 & F_0 \end{pmatrix}$$
, on peut simplifier et partir directement de:
$$\begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix}^n
= \begin{pmatrix} F_{n+1} & F_n \\ F_n & F_{n-1} \end{pmatrix}$$
Mais l'intérêt n'est pas évident, en effet la multiplication de matrices semble bien plus long à calculer que de simples additions.
Mais la puissance possède une propriété intéressante: $x^n$ avec $n=2k\implies x^{2k}=x^{2^k}$
On a du mal a voir l'intérêt, sauf si $k$ lui même est pair, donc: $k=2k'$, dans ce cas:
$\implies x^n=x^{2^k}=x^{2^{2^{k'}}}$. En bref, si on prend notre matrice, au carré, au carré, et ainsi de suite, on arrive à n en $O(\log_{2}(n))$ plutôt que $O(n)$. 

On peut maintenant faire une implémentation simple:
(Je n'utilise pas numpy car il ne supporte pas des nombres de plus de 64 bits, or la suite de Fibonnaci explose rapidement les compteurs 😀)

```python
import math
class Mat:
  def __init__(self, vals):
    self.mat = vals
    self.shape = (2,2)
  def mul(self, rhs):
    a,b,c,d = self.mat
    e,f,g,h = rhs.mat 
    return Mat([a*e+b*g, c*e+d*g, a*f+b*h, c*f+d*h])
  def exp(self, n):
    a = Mat(self.mat[:])
    for i in range(n):
      a = a.mul(self)
    return a
  def fast_exp(self, n):
    a = Mat(self.mat[:])
    base = Mat(self.mat[:])
    while n>0:
      if n&1:
        a = a.mul(base)
      base = base.mul(base)
      n >>= 1
    
    return a
def fib_mat(x): return Mat([1,1,1,0]).fast_exp(x).mat[3]

```
Faire de la vraie multiplication de matrices (donc plus que du 2*2) fonctionne très bien, mais je l'ai fait à la main pour réutiliser les valeurs, et c'est plus lisible.
$$\begin{pmatrix}
a & b \\
c & d
\end{pmatrix}*\begin{pmatrix}
e & f \\
g & h
\end{pmatrix}=\begin{pmatrix}
ae+bg & af+bh \\
ce+dg & cf+dh
\end{pmatrix}$$



### Formule explicite (Binet / De Moivre):
$$F_n = \frac{\left(\frac{1 + \sqrt{5}}{2}\right)^n - \left(\frac{1 - \sqrt{5}}{2}\right)^n}{\sqrt{5}}$$
Mais on peut voire que le terme de gauche diminue très rapidement, donc on peut simplifier (pour de très grand nombres): 
$$F_n = \left[\frac{\left(\frac{1 + \sqrt{5}}{2}\right)^n}{\sqrt{5}}\right]$$
```python
def fib_explicit(x):
  return int(((1+math.sqrt(5))/2)**x/(math.sqrt(5)))
```
Si on calcule l'erreur, elle peut être de plusieurs milliards ! Cela vient probablement de la précision de `math.sqrt(5)`.

Mais de toute manière, le pourcentage d'erreur reste faible (il diminue lorsque n augmente, et est proche de $10^{-14}$ pour des valeurs de $n=10_000$)

### Méthode de fast doubling

$$\begin{pmatrix}
F_{2k+1} & F_{2k}\\
F_{2k} & F_{2k-1}
\end{pmatrix}
=
\begin{pmatrix}
1 & 1\\
1 & 0
\end{pmatrix}^{2k}
=
\begin{pmatrix}
F_{k+1} & F_{k}\\
F_{k} & F_{k-1}
\end{pmatrix}
^2$$

$$\begin{align}
F_{2k+1} &= F_{k+1}^2 + F_{k}^2 \\
F_{2k} &= F_k(F_{k+1}+F_{k-1}) = F_k (2F_{k+1} - F_{k})\\
\end{align}.$$

```python
def fib_log2(n):
  if n==0:return 0,1
  fst,snd = fib_log2(n>>1)
  c = fst * (2 * snd - fst)
  d = fst**2+snd**2
  if n&1: return d,c+d
  return c,d

```








In [1]:
import math
def fib_naive(x):
  if x == 0 or x==1: return x
  return fib_naive(x-1)+fib_naive(x-2)
lut = {
	0: 0,
	1: 1
}

def fib_lut(x):
  if lut.get(x) != None: return lut[x]
  r = fib_lut(x-1)+fib_lut(x-2)
  lut[x] = r
  return r

def fib_iter(x):
  a = 0
  b = 1
  for i in range(x): 
    tmp = a + b
    a = b
    b = tmp
  return a;


class Mat:
  def __init__(self, vals):
    self.mat = vals
    self.shape = (2,2)
  def mul(self, rhs):
    a,b,c,d = self.mat
    e,f,g,h = rhs.mat 
    return Mat([a*e+b*g, c*e+d*g, a*f+b*h, c*f+d*h])
  def exp(self, n):
    a = Mat(self.mat[:])
    for i in range(n):
      a = a.mul(self)
    return a
  def fast_exp(self, n):
    a = Mat(self.mat[:])
    base = Mat(self.mat[:])
    while n>0:
      if n&1:
        a = a.mul(base)
      base = base.mul(base)
      n >>= 1
    
    return a
def fib_mat(x): return Mat([1,1,1,0]).fast_exp(x).mat[3]

def fib_explicit(x):
  return int(((1+math.sqrt(5))/2)**x/(math.sqrt(5)))

def fib_log2(n):
  if n==0:return 0,1
  fst,snd = fib_log2(n>>1)
  c = fst * (2 * snd - fst)
  d = fst**2+snd**2
  if n&1: return d,c+d
  return c,d

In [2]:
import time
def decorate(name, f, *args, **kwargs):
    start = time.time()
    f(*args, **kwargs)
    elapsed = time.time()-start
    if elapsed<1.:
        print(f"{name}: {elapsed*1000}ms for {args[0]} iterations")
    else:
        print(f"{name}: {elapsed}s for {args[0]} iterations")

In [ ]:
decorate("fib naive", fib_naive, 33)
import sys
sys.setrecursionlimit(150000)
decorate("fib lut", fib_lut, 100_000) # Not stable, got a lot of weird results
decorate("fib iter", fib_iter, 200_000) # ~As fast, a bit faster, uses less memory, so more scalable
decorate("fib mat", fib_mat, 1_000_000)
# decorate("fib explicit", fib_explicit, 1_000_000) # Python doesn't want to compute phi^1_000_000   😀
decorate("fib log2", fib_log2, 5_000_000)


fib naive: 590.5752182006836ms for 33 iterations
fib lut: 0.0030994415283203125ms for 100000 iterations
fib naive: 407.440185546875ms for 200000 iterations
fib naive: 548.3639240264893ms for 1000000 iterations
fib naive: 1.0278940200805664s for 5000000 iterations


Here are the results:

- fib naive: 590.5752182006836ms for 33 iterations
- fib lut: 0.0030994415283203125ms for 100000 iterations
- fib iter: 407.440185546875ms for 200000 iterations
- fib mat: 548.3639240264893ms for 1000000 iterations
- fib log2: 1.0278940200805664s for 5000000 iterations

